## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [3]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### masakhane/masakhaner2

In [6]:
label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-DATE": 7,
    "I-DATE": 8,
}

masakhaner2 = ner.ReadNERData()
masakhaner2_words, masakhaner2_labels = masakhaner2.read_dataset('masakhane/masakhaner2', label_map, lang='xho')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/5718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1633 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/1633 [00:00<?, ?it/s]

In [7]:
print(ner.check_labels(masakhaner2_labels))
label_alignment = {
    'I-PER': 'I-PER',
    'I-DATE': 'O',
    'B-ORG': 'B-ORG',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'O':     'O',
    'B-DATE': 'O',
    'B-PER': 'B-PER',
    'I-ORG': 'I-ORG',
}

# Align the dataset labels to the standard labels
masakhaner2_labels = ner.align_dataset(masakhaner2_labels, label_alignment)
print(ner.check_labels(masakhaner2_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'I-DATE', 'I-PER', 'B-LOC', 'B-DATE', 'I-LOC', 'B-ORG', 'B-PER', 'O', 'I-ORG'}
{'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'B-PER', 'O', 'I-ORG'}


# Evaluate model

In [8]:
alignment = {
'O':'O',
'B-DATE':'O',
'I-DATE':'O',
'B-PER':'B-PER',
'I-PER':'I-PER',
'B-ORG':'B-ORG',
'I-ORG':'I-ORG',
'B-LOC':'B-LOC',
'I-LOC':'I-LOC',
 }

model_name = "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
model_name_output = 'masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

tokenizer_config.json:   0%|          | 0.00/404 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [9]:
model_evaluation.model.config.id2label

{0: 'O',
 1: 'B-DATE',
 2: 'I-DATE',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

### masakhane/masakhaner2

In [10]:
data_name = "masakhane/masakhaner2"
masakhaner2_evaluation_output = model_evaluation.evaluate_model(masakhaner2_words, masakhaner2_labels)

  0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
masakhaner2_seqeval = masakhaner2_evaluation_output.get_classification('Seqeval')
masakhaner2_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8867,0.8998,0.8932,539
1,ORG,0.8241,0.9358,0.8764,841
2,PER,0.9622,0.9778,0.9700,1173
3,micro,0.8979,0.9475,0.9221,2553
4,macro,0.8910,0.9378,0.9132,2553
5,weighted,0.9008,0.9475,0.9229,2553


In [12]:
masakhaner2_sklearn = masakhaner2_evaluation_output.get_classification('Sklearn')
masakhaner2_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.9019,0.9086,0.9052,536
1,B-ORG,0.8502,0.9595,0.9016,840
2,B-PER,0.9656,0.9804,0.9729,1173
3,I-LOC,0.9129,0.8444,0.8773,360
4,I-ORG,0.8152,0.9451,0.8753,728
5,I-PER,0.9879,0.9852,0.9865,744
6,O,0.9969,0.9871,0.9920,21945
7,accuracy,0.9811,26326,None,None
8,macro,0.9186,0.9443,0.9301,26326
9,weighted,0.9825,0.9811,0.9815,26326
